## Running Flixtube Locally via Docker Compose

In this notebook, we look at how we can run the Flixtube microservice application locally using Docker Compose.

---

## Architecture

The `Flixtube` application is a microservice-based application with seven custom microservices:
  - `Web`: A Blazor frontend service with a user interface.
  - `Gateway`: An API gateway service, which the `Web` service talks to, and which redirects requests to the various other services below.
  - `Metadata`: A metadata service that manages video metadata (e.g. the name of a video).
  - `History`: A history service that manages video viewing history (e.g. the name, date and time of when a video was played).
  - `VideoUpload`: A service used to upload a video from the `Web` frontend via the `Gateway` to the `VideoStorage` service.
  - `VideoStorage`: A service responsible for storing video files.
  - `Videostreaming`: A service used to stream a video from the `VideoStorage` service via the `Gateway` to the `Web` service.

The `Flixtube` application uses a number of **backing services**:
  - `Sql Server`: An SQL Server service to store relational data.
  - `Minio` or `Azure Storage`: A Minio or Azure Storage service to store Blob data (video files).
  - `Rabbit MQ`: A Rabbit Message Queue (MQ) service for publisher/subscriber-based messaging.

<img src="notebook_images/flixtube_architecture.drawio.png"  width="800" alt="Flixtube Architecture" />

**Communication**

- The **Browser** communicates with:
  - The `Web` microservice via HTTP.
  - The `Gateway` microservice via HTTP when streaming (playing) a Video.
- The `Gateway` communicates with the microservices below via HTTP:
  - `Web`, `Metadata`, `History`, `VideoStreaming`, `VideoUpload`, and `VideoStorage`.
- `VideoStorage` communicates with:
  - The `Gateway`, `VideoStreaming`, and `VideoUpload` microservices via HTTP.
  - A storage backing service which can be one of the below:
    - `Minio` via HTTP if the `VideoStorage` microservice is built using the `MinioStorage` VSCode project.
    - `Azure Storage` via HTTP if the `VideoStorage` microservice is built using the `AzureStorage` VSCode project.
- `VideoUpload` communicates with:
  - The `Gateway` and `VideoStorage` microservices via HTTP.
  - A `RabbitMQ` backing service via AMQP, where it publishes `VideoUploaded` messages to an Exchange called `uploaded`.
- `VideoStreaming` communicates with:
  - The `Gateway` and `VideoStorage` microservices via HTTP.
  - A `RabbitMQ` backing service via AMQP, where it publishes `VideoViewed` messages to an Exchange called `viewed`.
- `Metadata` communicates with:
  - The `Gateway` microservice via HTTP.
  - An `SQLServer` backing service via HTTP, where it manages video metadata in a `Videos` table. 
  - A `RabbitMQ` backing service via AMQP, where it subscribes to an Exchange called `uploaded` receiving `VideoUploaded` messages.
- `History` communicates with:
  - The `Gateway` microservice via HTTP.
  - An `SQLServer` backing service via HTTP, where it manages video viewing history in a `ViewHistorys` table. 
  - A `RabbitMQ` backing service via AMQP, where it subscribes to an Exchange called `viewed` receiving `VideoViewed` messages.

---

## Running Flixtube via Docker Compose

### Run the command below to bring up the microservices using Docker Compose

- Command: `docker compose -f <PathToYAMLFile> --project-directory <PathToFolder> up -d --build --force-recreate`
  - `<PathToYAMLFile>` relative to this notebook is `../flixtube/compose/docker-compose-web-dev.yml`
  - `<PathToFolder>` relative to this notebook is `../flixtube`

**Note**

- The folder `../flixtube/compose` contains numerous Docker Compose YAML file that might be useful:
  - `docker-compose-azure-storage-dev.yml` brings up the `AzureStorage` microservice with all its dependencies.
  - `docker-compose-gateway-dev.yml` brings up the `Gateway` microservice with all its dependencies.
  - `docker-compose-history-dev.yml` brings up the `History` microservice with all its dependencies.
  - `docker-compose-metadata-dev.yml` brings up the `Metadata` microservice with all its dependencies.
  - `docker-compose-minio-storage-dev.yml` brings up the `MinioStorage` microservice with all its dependencies.
  - `docker-compose-video-streaming-dev.yml` brings up the `VideoStreaming` microservice with all its dependencies.
  - `docker-compose-video-upload-dev.yml` brings up the `VideoUpload` microservice with all its dependencies.
  - `docker-compose-web-dev.yml` brings up the `Web` microservice with all its dependencies (development containers).
  - `docker-compose-web-prod.yml` brings up the `Web` microservice with all its dependencies (production containers).
- Only run either the `AzureStorage` or the `MinioStorage` microservice at a time.
  - `AzureStorage` is for storing video files in the Azure cloud.
  - `Minio` is for storing video files locally during development to avoid the cost of using Azure storage.
- If you want to run the entire application, use `docker-compose-web-dev.yml` or `docker-compose-prod-dev.yml`
  - `docker-compose-web-dev.yml` builds the development containers with hot reload enabled and the code files shared with the host.
  - `docker-compose-prod-dev.yml` builds the production containers with hot reload disabled and the code files inside the image.

In [1]:
!docker compose -f ../flixtube/compose/docker-compose-web-dev.yml --project-directory ../flixtube up -d --build --force-recreate

#0 building with "desktop-linux" instance using docker driver

#1 [minio-storage internal] load build definition from Dockerfile-dev
#1 transferring dockerfile: 300B 0.0s done
#1 DONE 0.0s

#2 [metadata internal] load build definition from Dockerfile-dev
#2 transferring dockerfile: 454B 0.0s done
#2 DONE 0.1s

#3 [history internal] load build definition from Dockerfile-dev
#3 transferring dockerfile: 285B done
#3 DONE 0.1s

#4 [history internal] load metadata for mcr.microsoft.com/dotnet/sdk:9.0
#4 DONE 0.4s

#5 [history internal] load .dockerignore
#5 transferring context: 393B done
#5 DONE 0.0s

#6 [minio-storage internal] load .dockerignore
#6 transferring context: 393B done
#6 DONE 0.0s

#7 [metadata 1/5] FROM mcr.microsoft.com/dotnet/sdk:9.0@sha256:84fd557bebc64015e731aca1085b92c7619e49bdbe247e57392a43d92276f617
#7 DONE 0.0s

#8 [metadata internal] load .dockerignore
#8 transferring context: 393B done
#8 DONE 0.0s

#9 [metadata internal] load build context
#9 transferring context:

 Service minio-storage  Building
 Service metadata  Building
 Service history  Building
 Service minio-storage  Built
 Service video-streaming  Building
 Service metadata  Built
 Service video-upload  Building
 Service history  Built
 Service video-streaming  Built
 Service video-upload  Built
 Service gateway  Building
 Service gateway  Built
 Service web  Building
 Service web  Built
 Network flixtube_default  Creating
 Network flixtube_default  Created
 Volume "flixtube_rabbit_data"  Creating
 Volume "flixtube_rabbit_data"  Created
 Volume "flixtube_minio_data"  Creating
 Volume "flixtube_minio_data"  Created
 Volume "flixtube_sqlserver_data"  Creating
 Volume "flixtube_sqlserver_data"  Created
 Container rabbit  Creating
 Container sqlserver  Creating
 Container minio  Creating
 Container sqlserver  Created
 Container minio  Created
 Container video-storage  Creating
 Container minio-mc  Creating
 Container rabbit  Created
 Container metadata  Creating
 Container history  Creating


### List all Images on the Local Computer

- Notice that all the images have been built by Docker Compose.
  - `web`, `gateway`, `video-streaming`, `video-upload`, `history`, `metadata`, and `minio-storage`.
- Our backing services have also been downloaded.
  - `rabbitmq`, `sqlserver`, and `minio`.

In [14]:
!docker images

REPOSITORY                                           TAG                                                                           IMAGE ID       CREATED         SIZE
web                                                  latest                                                                        21245537ff52   12 days ago     4.17GB
gateway                                              latest                                                                        8ffb6a4baf0b   13 days ago     2.01GB
history                                              latest                                                                        a382c35718e9   13 days ago     1.33GB
video-streaming                                      latest                                                                        532b343da17a   13 days ago     834MB
video-upload                                         latest                                                                        24bc62559f32   13 days ago 

### List all running Containers on the Local Computer

- Notice that all the containers are up and running.
  - Our own microservices: `web`, `gateway`, `video-streaming`, `video-upload`, `history`, `metadata`, and `minio-storage`.
  - Our backing services: `rabbitmq`, `sqlserver`, and `minio`.
- Notice the port forward of the host's port `4000` to the `web` microservice's port `80` (http://localhost:4000).
- Notice the port forward of the host's port `4010` to the `gateway` microservice's port `80` (http://localhost:4010).
- Notice the port forward of the host's port `9001` to the `minio` backing service's port `9001` (http://localhost:9001).
  - This is to Minio's Web UI for managing its object storage (the other port `9000` is its data port).
- Notice the port forward of the host's port `15672` to the `rabbitmq` backing service's port `15672` (http://localhost:15672).
  - This is to RabbitMQ's Web UI for managing its exchanges, queues, and messages (the other port `5672` is its data port).

In [ ]:
!docker ps

CONTAINER ID   IMAGE                                        COMMAND                  CREATED         STATUS         PORTS                                                                                                         NAMES
a6f454c2fa56   web                                          "dotnet watch run --…"   4 minutes ago   Up 4 minutes   0.0.0.0:4000->80/tcp                                                                                          web
d5dbe2cc6cb0   gateway                                      "dotnet watch run --…"   4 minutes ago   Up 4 minutes   0.0.0.0:4010->80/tcp                                                                                          gateway
717ee4e8ceb1   video-streaming                              "dotnet watch run --…"   4 minutes ago   Up 4 minutes   0.0.0.0:4040->80/tcp                                                                                          video-streaming
e1bd2b92bda1   video-upload                                 "d

---

## Using the Flixtube Application

- Visit the Home page http://localhost:4000

  - The Home page displays a list of videos (currently empty) with an `Id` and a `Name` column.
  - There are two buttons `Upload Video` and `Show Viewing History` at the top of the page.
    - The `Upload Video` button is used for uploading a video.
    - The `Show Viewing History` button is used to display a list of viewing history, i.e. when a video was viewed (played).
  - Click the `Upload Video` button on the Home page which will navigate to the Upload Video page.

<img src="notebook_images/flixtube_home_page.png"  width="600" alt="Flixtube Home Page" />

- The Upload Video page.

  - The Upload Video page allows the user to upload a video from the user's local file system.
  - There are four buttons `Home`, `Choose File`, `Upload` and `Cancel` on the page.
    - The `Home` button navigates back to the Home page.
    - The `Choose File` button pops up a window, allowing the user to choose a video file to upload.
      - The name of the chosen video file is displayed in the field to the right of the `Choose File` button.
    - The `Upload` button uploads the chosen video file.
    - The `Cancel` button returns the user to the Home page without uploading any video file.
  - Click the `Choose File` button and choose a video file.
    - There is a sample file in `workshop3/SampleVideo_1280x720_1mb.mp4` you can use.
  - Then click the `Upload` button, which will upload the file and navigate back to the Home page.

<img src="notebook_images/flixtube_upload_page.png"  width="600" alt="Flixtube Upload Video Page" />

- Back on the Home page.
  - The Home page now displays one video in its a list of videos.
    - The `Id` column contains the video's GUID.
    - The `Name` column contains the video's name.
    - The blue `Play` button is used to play (view) the video.
    - The red `Delete` button is used to remove (delete) the video.
  - We could also access the Gateway to get the JSON document via the REST API.
    - http://localhost:4010/api/metadata
- Click the blue `Play` button which will navigate to the Play Video page.

<img src="notebook_images/flixtube_home_page2.png"  width="600" alt="Flixtube Home Page" />

- The Play Video page.

  - The Play Video page starts streaming the video to the browser.
    - The video's name is shown above the video.
    - Video controls for playing, pausing, etc. are shown below the video.
    - There is a `Home` button at the top of the page, which will navigate back to the Home page.
  - Click the `Home` button to navigate back to the Home page.

<img src="notebook_images/flixtube_play_page.png"  width="600" alt="Flixtube Play Video Page" />

- Back on the Home page.

  - Click the `Show Viewing History` button which will navigate to the Viewing History page.

<img src="notebook_images/flixtube_home_page2.png"  width="600" alt="Flixtube Home Page" />

- The Viewing History page.
  - The Viewing History page displays a list video viewings with an `Id` and a `ViewedAt` column.
    - The `Id` column contains the video's GUID.
    - The `ViewedAt` column contains the date and time the video was viewed (played).
    - There is a `Home` button at the top of the page, which will navigate back to the Home page.
  - We could also access the Gateway to get the JSON document via the REST API.
    - http://localhost:4010/api/history
  - Click the `Home` button to navigate back to the Home page.

<img src="notebook_images/flixtube_history_page.png"  width="600" alt="Flixtube Viewing History Page" />

---

## Using Minio's Web UI

- Visit http://localhost:9001
- Login with **username** `minio_user` and **password** `minio_password`
- Under `User`, choose `Object Browser`, then click the `videos` bucket.
  - In the `videos` bucket, you will see the video files stored in the bucket (upload one via the `web` if it's empty).
- Under `Administrator`, choose `Buckets`, then click the `videos` bucket.
  - You will see the settings for the `video` bucket, with `Access Policy` set to `Public` (anyone can use the bucket).
  - Click on the `Access`, the select the `Users` tab.
  - Click on the user `Y42xk0CJlYV2fnShCtBP`, then select `Policies`.
  - Under `Policies`, notice the user has `readwrite` access to the bucket (i.e. can upload and download files). 
- All these settings were configured by the `minio-mc` service (see the Docker Compose YAML file).
  - It uses the script `minio_initialize.sh` in the folder `flixtube/fixtures` to do this.
  - This service is started as a container, but will exit right away after it has configured Minio.

You can use this Web UI during development to manage various Minio objects, e.g. buckets, users, etc.

---

## Using RabbitMQ's Web UI

- Visit: http://localhost:15672
- Login with **username** `rabbit_user` and **password** `rabbit_password`
- Under the `Exchanges` tab, in the `Name` column, notice the `uploaded` and `viewed` exchanges.
- Click the `uploaded` or `viewed` exchange.
  - Under `Publish message`, you can enter a JSON document in the `Payload` field and click the `Publish message` button.
  - This will publish a message to the exchange.

You can use this Web UI during development to manage various RabbitMQ objects, e.g. messages, exchanges, queues, etc.

---

## Stopping and Removing Flixtube via Docker Compose

### Run the command below to tear down the microservices using Docker Compose

- Command: `docker compose -f <PathToYAMLFile> --project-directory <PathToFolder> down --rmi local --volumes`
  - `<PathToYAMLFile>` relative to this notebook is `../flixtube/compose/docker-compose-web-dev.yml`
  - `<PathToFolder>` relative to this notebook is `../flixtube`

In [15]:
!docker compose -f ../flixtube/compose/docker-compose-web-dev.yml --project-directory ../flixtube down --rmi local --volumes

 Container minio-mc  Stopping
 Container web  Stopping
 Container minio-mc  Stopped
 Container minio-mc  Removing
 Container minio-mc  Removed
 Container web  Stopped
 Container web  Removing
 Container web  Removed
 Container gateway  Stopping
 Container gateway  Stopped
 Container gateway  Removing
 Container gateway  Removed
 Container history  Stopping
 Container video-streaming  Stopping
 Container video-upload  Stopping
 Container metadata  Stopping
 Container video-streaming  Stopped
 Container video-streaming  Removing
 Container history  Stopped
 Container history  Removing
 Container metadata  Stopped
 Container metadata  Removing
 Container video-upload  Stopped
 Container video-upload  Removing
 Container video-streaming  Removed
 Container metadata  Removed
 Container video-upload  Removed
 Container video-storage  Stopping
 Container history  Removed
 Container rabbit  Stopping
 Container sqlserver  Stopping
 Container rabbit  Stopped
 Container rabbit  Removing
 Containe

### List all Containers on the Local Computer

- Notice that all the containers have been stopped and removed.
  - Our own microservices: `web`, `gateway`, `video-streaming`, `video-upload`, `history`, `metadata`, and `minio-storage`.
  - Our backing services: `rabbitmq`, `sqlserver`, and `minio`.

In [ ]:
!docker ps -a

CONTAINER ID   IMAGE          COMMAND                  CREATED        STATUS        PORTS     NAMES


### List all Images on the Local Computer

- Notice that all **our own** images have been removed by Docker Compose.
  - `web`, `gateway`, `video-streaming`, `video-upload`, `history`, `metadata`, and `minio-storage`.
- Notice that the images for the backing services (`rabbitmq`, `sqlserver`, `minio`) have not been removed.

In [18]:
!docker images

REPOSITORY                                           TAG                                                                           IMAGE ID       CREATED         SIZE
mcr.microsoft.com/mssql/server                       2022-latest                                                                   b823451808db   2 weeks ago     1.62GB
minio/minio                                          RELEASE.2024-10-13T13-34-11Z                                                  fc911b10a206   3 months ago    165MB
rabbitmq                                             4.0.2-management                                                              db27bd782e92   4 months ago    256MB
